# Stage1 0bp 首小时方向成交量与局部净恢复价值

先逐单验证旧 A_0 Tiered 费用实现；通过后独立审阅同状态恢复/HOLD 标签代码，再进行唯一固定信息检验。

In [1]:
import os,json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.backtest.ibkr_tiered import TieredFees
root=Path.cwd();card=root/'research/stage1_0bp_signed_volume_local_value_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen']
ps=plan['parent'];assert sha(root/ps['summary'])==ps['summary_sha256'];ss=json.loads((root/ps['summary']).read_text());assert ss['recorder_id']==ps['recorder_id'] and ss['nas_evidence']['sha256']==ps['NAS_archive_sha256']
readback=json.loads((root/Path(ps['summary']).parent/'readback_verification.json').read_text());assert readback['status']=='passed' and readback['archive_sha256']==ps['NAS_archive_sha256'] and readback['all_artifact_hashes_match']
art=Path(ps['recorder_uri'])/ps['experiment_id']/ps['recorder_id']/'artifacts'
for name,digest in ps['artifact_sha256'].items():assert sha(art/name)==digest
assert sha(root/plan['fee_runtime']['module'])==plan['fee_runtime']['current_module_sha256'] and sha(art/'sources/src__backtest__ibkr_tiered.py')==plan['fee_runtime']['frozen_source_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(ps['recorder_uri']):
 r=R.get_recorder(recorder_id=ps['recorder_id'],experiment_id=ps['experiment_id']);native=r.load_object('native/A_active_research_replay_0');parent_plan=r.load_object('inputs/plan')
orders=native['orders'].reset_index(drop=True);assert len(orders)==plan['fee_runtime']['observed_order_count'] and pd.to_datetime(orders.stamp).is_monotonic_increasing
fees=TieredFees(parent_plan['tariff'],slippage_bps=0);fields=['brokerage','venue','clearing','sec','taf','cat','pass_through','slippage','notional','total'];maxdiff={k:0.0 for k in fields};fail=[];capmismatch=0
for i,o in orders.iterrows():
 quote=fees.quote(shares=int(o.raw_shares),raw_price=float(o.raw_price),sell=bool(o.sell),stamp=o.stamp)
 for key in fields:
  diff=abs(float(o[key])-float(quote[key]));maxdiff[key]=max(maxdiff[key],diff)
  if diff>plan['fee_runtime']['max_fee_field_abs_error'] and len(fail)<12:fail.append({'order_index':int(i),'field':key,'observed':float(o[key]),'replayed':float(quote[key]),'abs_diff':diff})
 if bool(o.cap_applied)!=bool(quote['cap_applied']):capmismatch+=1
 fees.commit(int(o.raw_shares),stamp=o.stamp,cap_applied=bool(quote['cap_applied']))
passed=not fail and not capmismatch and max(maxdiff.values())<=plan['fee_runtime']['max_fee_field_abs_error']
result={'status':'A0_0bp_observed_Tiered_fee_parity_passed' if passed else 'A0_0bp_observed_Tiered_fee_parity_failed_stop_before_labels','plan_sha256':sha(card/'plan.json'),'observed_orders':len(orders),'checked_fee_fields':fields,'max_abs_diff_by_field':maxdiff,'cap_flag_mismatches':capmismatch,'first_failures':fail,'final_monthly_share_counters':fees.monthly_shares,'current_module_equals_frozen_source_object_bytes':plan['fee_runtime']['current_module_sha256']==plan['fee_runtime']['frozen_source_artifact_sha256'],'new_model_fits':0,'new_native_accounts':0,'local_value_labels_computed':False}
receipt=card/'fee_parity_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==result
else:receipt.write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','observed_orders','cap_flag_mismatches','first_failures')});assert passed


2026/10/23 04:30:36 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[82018:MainThread](2026-10-23 04:30:36,995) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[82018:MainThread](2026-10-23 04:30:36,996) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[82018:MainThread](2026-10-23 04:30:36,996) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


{'status': 'A0_0bp_observed_Tiered_fee_parity_passed', 'observed_orders': 1275, 'cap_flag_mismatches': 0, 'first_failures': []}


## 局部标签执行门

费用等价和准确标签代码需独立审阅；未批准前不读取次日结算价。

In [2]:
import os,json,hashlib,copy,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from scipy.stats import rankdata
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.backtest.ibkr_tiered import TieredFees
from src.backtest.raw_corporate_policy import settle_raw
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_0bp_signed_volume_local_value_20261023';plan=json.loads((card/'plan.json').read_text());add=json.loads((card/'label_execution_addendum.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
review_path=card/'design_review_final.json';assert review_path.exists();review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());fee_code=''.join(notebook['cells'][1]['source']);fee_code_hash=hashlib.sha256(fee_code.encode()).hexdigest();code=''.join(notebook['cells'][3]['source']);code_hash=hashlib.sha256(code.encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['base_plan_sha256']==sha(card/'plan.json') and review['addendum_sha256']==sha(card/'label_execution_addendum.json') and review['raw_binding_sha256']==sha(card/'raw_settlement_binding_addendum.json') and review['fee_parity_sha256']==sha(card/'fee_parity_result.json') and review['projection_parity_sha256']==sha(card/'projection_parity_result.json') and review['fee_cell_source_sha256']==fee_code_hash and review['label_cell_source_sha256']==code_hash
assert add['base_plan_sha256']==sha(card/'plan.json') and add['fee_parity_result_sha256']==sha(card/'fee_parity_result.json') and add['executed_projection_parity_sha256']==sha(card/'projection_parity_result.json')
raw_binding=json.loads((card/'raw_settlement_binding_addendum.json').read_text());assert raw_binding['base_plan_sha256']==sha(card/'plan.json') and raw_binding['label_execution_addendum_sha256']==sha(card/'label_execution_addendum.json') and sha(root/raw_binding['raw_settlement_module'])==raw_binding['raw_settlement_module_sha256']
fee_result=json.loads((card/'fee_parity_result.json').read_text());projection_result=json.loads((card/'projection_parity_result.json').read_text());assert fee_result['status']==add['fee_parity_status_required'] and projection_result['status']==add['projection_parity_status_required']
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'status':'started','base_plan_sha256':sha(card/'plan.json'),'addendum_sha256':sha(card/'label_execution_addendum.json'),'label_cell_source_sha256':code_hash,'new_model_fits':0,'new_native_accounts':0,'bootstrap_attempted':False};state_path.write_text(json.dumps(state,indent=2)+'\n')
ps=plan['parent'];rs=plan['source_route'];assert sha(root/ps['summary'])==ps['summary_sha256'] and sha(root/rs['summary'])==rs['summary_sha256']
pa=Path(ps['recorder_uri'])/ps['experiment_id']/ps['recorder_id']/'artifacts';ra=Path(rs['recorder_uri'])/rs['experiment_id']/rs['recorder_id']/'artifacts'
for name,digest in ps['artifact_sha256'].items():assert sha(pa/name)==digest
assert sha(ra/rs['daily_artifact'])==rs['daily_artifact_sha256'];assert sha(root/plan['fee_runtime']['module'])==plan['fee_runtime']['current_module_sha256'] and sha(root/'src/workflow/intraday_recovery_direct_allocation.py')==json.loads((card/'projection_parity_plan.json').read_text())['projection_module_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2023-01-04',end_time='2026-10-08',freq='day'));assert len(calendar)==944
with R.uri_context(ps['recorder_uri']):
 rec=R.get_recorder(recorder_id=ps['recorder_id'],experiment_id=ps['experiment_id']);native=rec.load_object('native/A_active_research_replay_0');frame=rec.load_object('inputs/frame');parent_plan=rec.load_object('inputs/plan');selection=rec.load_object('inputs/selection');recovery=rec.load_object('inputs/recovery_spec');schedule=rec.load_object('inputs/schedule')
with R.uri_context(rs['recorder_uri']):route=R.get_recorder(recorder_id=rs['recorder_id'],experiment_id=rs['experiment_id']).load_object(rs['daily_artifact'])
a=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True);points=native['points'];orders=native['orders'].reset_index(drop=True)
for x in (a,targets,frame,route):x['trade_date']=pd.to_datetime(x.trade_date).dt.normalize()
assert len(a)==len(targets)==len(frame)==len(route)==len(calendar)==944 and a.trade_date.equals(frame.trade_date) and a.trade_date.equals(targets.trade_date) and a.trade_date.equals(route.trade_date) and a.trade_date.equals(pd.Series(calendar,name='trade_date'))
assert not a.recovery_action.any() and int(route.necessary_route.sum())==rs['necessary_route_dates'] and int(route.necessary_route_source_known.sum())==rs['source_known_dates']
point1032=points.loc[pd.to_datetime(points.stamp).dt.strftime('%H:%M').eq('10:32')].copy();point1032['trade_date']=pd.to_datetime(point1032.stamp).dt.normalize();point1032=point1032.set_index('trade_date');assert len(point1032)==944 and point1032.index.equals(calendar)
point0947=points.loc[pd.to_datetime(points.stamp).dt.strftime('%H:%M').eq('09:47')].copy();point0947['trade_date']=pd.to_datetime(point0947.stamp).dt.normalize();point0947=point0947.set_index('trade_date')
monthly=TieredFees(parent_plan['tariff'],slippage_bps=0);order_at=pd.to_datetime(orders.stamp);order_i=0;daily=[];eligible_count=0;label_count=0;corporate_unknown=0;last_unknown=0;max_current_nav_gap=0.;max_hold_q_gap=0.;max_hold_cash_gap=0.;max_hold_nav_gap=0.
for i in range(944):
 row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=10,minutes=32)
 while order_i<len(orders) and order_at.iloc[order_i]<stamp:
  order=orders.iloc[order_i];monthly.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied));order_i+=1
 point=point1032.loc[day];q=point[['q_soxl','q_soxx']].to_numpy(float);cash=float(point.cash);receivable=float(point.receivable);price=row[['raw_soxl_recovery','raw_soxx_recovery']].to_numpy(float)
 current_nav=float(q@price+cash+receivable);gap=abs(current_nav-float(point.nav));max_current_nav_gap=max(max_current_nav_gap,gap);assert gap<=0.01
 source_pressure=float(route.iloc[i].pressure) if bool(route.iloc[i].source_known) else np.nan;necessary=bool(route.iloc[i].necessary_route);value_bp=np.nan;label_at=pd.NaT;status='not_necessary_route'
 if necessary:
  eligible_count+=1;assert np.isfinite(source_pressure)
  if i+1>=944:
   last_unknown+=1;status='unknown_no_next_Stage1_trade_date'
  else:
   nxt=frame.iloc[i+1];nextday=pd.Timestamp(nxt.trade_date)
   if nextday in schedule:
    corporate_unknown+=1;status='unknown_next_date_corporate_action'
   else:
    assert float(route.iloc[i].pressure)==source_pressure
    raw_beta=3*float(row.soxl_weight)+float(row.soxx_weight);beta=min(raw_beta,float(selection['action_beta']))
    assert beta>float(a.iloc[i].current_beta_cap)+1e-12
    target=project_legal_weights(float(row.soxl_weight),float(row.soxx_weight),beta,minimum_cash_weight=float(recovery['minimum_cash_weight']),maximum_soxl_weight=float(recovery['maximum_soxl_weight']))
    nextprice=nxt[['raw_soxl_open','raw_soxx_open']].to_numpy(float);nextstamp=nextday+pd.Timedelta(hours=9,minutes=47)
    next_target=targets.iloc[i+1][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float) if bool(a.iloc[i+1].execute_open_target) else None
    branch_values=[];branch_results=[]
    for desired in (target,None):
     branch_fee=copy.deepcopy(monthly);entry_q,entry_cash,_=settle_raw(q,cash,receivable,desired=desired,prices=price,stamp=stamp,fees=branch_fee)
     reset_q,reset_cash,_=settle_raw(entry_q,entry_cash,receivable,desired=next_target,prices=nextprice,stamp=nextstamp,fees=branch_fee)
     ending=float(reset_q@nextprice+reset_cash+receivable);branch_values.append(ending);branch_results.append((reset_q,reset_cash))
    baseline_q,baseline_cash=branch_results[1];saved=point0947.loc[nextday]
    q_gap=float(np.max(np.abs(baseline_q-saved[['q_soxl','q_soxx']].to_numpy(float))));cash_gap=abs(float(baseline_cash)-float(saved.cash));nav_gap=abs(branch_values[1]-float(saved.nav))
    max_hold_q_gap=max(max_hold_q_gap,q_gap);max_hold_cash_gap=max(max_hold_cash_gap,cash_gap);max_hold_nav_gap=max(max_hold_nav_gap,nav_gap)
    assert q_gap<=1e-8 and cash_gap<=0.02 and nav_gap<=0.02, f'HOLD branch A0 reset parity failed {day}: q={q_gap} cash={cash_gap} nav={nav_gap}'
    value_bp=(branch_values[0]-branch_values[1])/current_nav*10000;assert np.isfinite(value_bp)
    label_at=nextstamp+pd.Timedelta(minutes=2);status='mature_local_value_label';label_count+=1
 daily.append({'trade_date':day,'source_pressure':source_pressure,'necessary_route':necessary,'source_known':bool(route.iloc[i].source_known),'status':status,'label_available_at':label_at,'local_recovery_minus_HOLD_bp':value_bp,'current_nav':current_nav,'current_receivable':receivable,'risk_state':int(a.iloc[i].risk_state),'current_beta_cap':float(a.iloc[i].current_beta_cap)})
labels=pd.DataFrame(daily);assert len(labels)==944 and labels.trade_date.equals(pd.Series(calendar,name='trade_date')) and int(labels.necessary_route.sum())==199
assert label_count+corporate_unknown+last_unknown==199 and label_count>=add['minimum_mature_labels']['full']
halves=[labels.loc[labels.trade_date.lt(pd.Timestamp('2025-01-01'))],labels.loc[labels.trade_date.ge(pd.Timestamp('2025-01-01'))]]
assert sum(x.local_recovery_minus_HOLD_bp.notna().sum() for x in halves)==label_count and halves[0].local_recovery_minus_HOLD_bp.notna().sum()>=add['minimum_mature_labels']['first_half'] and halves[1].local_recovery_minus_HOLD_bp.notna().sum()>=add['minimum_mature_labels']['second_half']
def rank_corr(x,y):
 mask=np.isfinite(x)&np.isfinite(y)
 if mask.sum()<3:return np.nan
 xr=rankdata(x[mask]);yr=rankdata(y[mask]);return float(np.corrcoef(xr,yr)[0,1]) if np.std(xr)>0 and np.std(yr)>0 else np.nan
x=labels.source_pressure.to_numpy(float);y=labels.local_recovery_minus_HOLD_bp.to_numpy(float);full=rank_corr(x,y);half=[rank_corr(z.source_pressure.to_numpy(float),z.local_recovery_minus_HOLD_bp.to_numpy(float)) for z in halves];assert np.isfinite(full) and np.isfinite(half).all()
boot=plan['information_gate']['stationary_calendar_bootstrap'];rng=np.random.default_rng(boot['seed']);n=len(labels);reps=np.full(boot['draws'],np.nan)
state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
for b in range(boot['draws']):
 idx=np.empty(n,dtype=np.int32);pos=int(rng.integers(n));idx[0]=pos
 for t in range(1,n):pos=int(rng.integers(n)) if rng.random()<1/boot['mean_block_sessions'] else (pos+1)%n;idx[t]=pos
 reps[b]=rank_corr(x[idx],y[idx])
valid=reps[np.isfinite(reps)];assert len(valid)>=9900
lower=float(np.quantile(valid,boot['one_sided_alpha']));upper=float(np.quantile(valid,1-boot['one_sided_alpha']))
gates={'full_positive':full>0,'both_halves_positive':all(v>0 for v in half),'bootstrap_lower_positive':lower>0};passed=bool(all(gates.values()))
quarters=[]
for k,g in labels.groupby(labels.trade_date.dt.to_period('Q'),sort=True):
 v=g.local_recovery_minus_HOLD_bp;quarters.append({'quarter':str(k),'sessions':len(g),'necessary_route':int(g.necessary_route.sum()),'mature_labels':int(v.notna().sum()),'mean_local_value_bp':float(v.mean()) if v.notna().any() else None,'pressure_value_spearman':rank_corr(g.source_pressure.to_numpy(float),v.to_numpy(float)) if v.notna().sum()>=10 else None})
static=pd.concat({'feature':labels.set_index('trade_date')[['source_pressure']],'label':labels.set_index('trade_date')[['local_recovery_minus_HOLD_bp']],'meta':labels.set_index('trade_date')[['necessary_route','source_known','label_available_at']]},axis=1);static.index=pd.MultiIndex.from_arrays([labels.trade_date,['SOXL_SOXX_CASH']*len(labels)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':('2023-01-04','2026-10-08')});prepared=dataset.prepare('held',col_set=['feature','label','meta']);assert len(prepared)==944
summary={'experiment':card.name,'plan_sha256':sha(card/'plan.json'),'addendum_sha256':sha(card/'label_execution_addendum.json'),'decision':'conditional_local_0bp_signed_volume_information_pass_source_clock_pending_no_account' if passed else 'not_promoted_fixed_0bp_signed_volume_local_value_rank_failed','objective_achieved':False,'matched_sessions':944,'necessary_route_days':199,'mature_local_labels':label_count,'unknown_next_corporate_action':corporate_unknown,'unknown_no_next_date':last_unknown,'fee_parity_observed_orders':fee_result['observed_orders'],'executed_legal_projection_parity_rows':projection_result['executed_rows'],'max_current_physical_nav_gap_usd':max_current_nav_gap,'max_HOLD_next_reset_quantity_gap':max_hold_q_gap,'max_HOLD_next_reset_cash_gap_usd':max_hold_cash_gap,'max_HOLD_next_reset_nav_gap_usd':max_hold_nav_gap,'full_pressure_value_spearman':full,'half_pressure_value_spearman':half,'bootstrap_lower':lower,'bootstrap_upper':upper,'bootstrap_valid_replicates':len(valid),'gates':gates,'information_gate_pass':passed,'quarters':quarters,'historical_independence':False,'source_first_receipt_finality_qualified':False,'new_model_fits':0,'new_native_accounts':0,'local_labels_are_full_strategy_value':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/bootstrap_replicates']}}
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,addendum=add,dataset=dataset,fee_parity=fee_result,projection_parity=projection_result)
 rec.save_objects(artifact_path='analysis',daily=labels,bootstrap_replicates=reps,quarters=quarters,gates=gates)
 rec.log_metrics(mature_local_labels=label_count,full_pressure_value_spearman=full,bootstrap_lower=lower,information_gate_pass=int(passed),new_model_fits=0,new_native_accounts=0)
 with R.uri_context(str(tracking)):readback=R.get_recorder(recorder_id=rec.id,experiment_id=str(rec.experiment_id)).load_object('analysis/daily')
 assert readback.equals(labels)
summary.update(recorder_id=state['recorder_id'],experiment_id=state['experiment_id'],recorder_uri=str(tracking));(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='stage1-local-value-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[m.name for m in source.getmembers() if m.isfile()];assert any(x.endswith('/artifacts/analysis/daily') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'mature_labels':label_count,'full_spearman':full,'lower':lower,'new_accounts':0})


[82018:MainThread](2026-10-23 04:30:38,310) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[82018:MainThread](2026-10-23 04:30:38,311) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[82018:MainThread](2026-10-23 04:30:38,311) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=10,minutes=32)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=10,minutes=32)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes impli

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=10,minutes=32)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=10,minutes=32)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_82018/861740865.py:39: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes impli

...

[82018:MainThread](2026-10-23 04:30:42,389) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_0bp_signed_volume_local_value_20261023.


[82018:MainThread](2026-10-23 04:30:42,391) INFO - qlib.workflow - [exp.py:258] - Experiment 271674762291940481 starts running ...


[82018:MainThread](2026-10-23 04:30:42,469) INFO - qlib.workflow - [recorder.py:345] - Recorder 7abbbcf0fcf145b69df17ee77624aa80 starts running under Experiment 271674762291940481 ...


[82018:MainThread](2026-10-23 04:30:43,111) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'not_promoted_fixed_0bp_signed_volume_local_value_rank_failed', 'mature_labels': 191, 'full_spearman': 0.09892360154312481, 'lower': -0.05159080748760015, 'new_accounts': 0}


# 原始 Stage1 0bp 首小时带方向成交量与局部净恢复价值

结论：同一带方向成交量机制的固定 2018–2022 OOF 训练/评价信息检验已失败；本卡在原始 Stage1 A_0 的 2023–2026 复用历史上补充实物账户的局部动作价值归因，也未通过校正统计门。该路线关闭，不调符号、10:31 时点、beta、0.825 门槛或选择季度重试。

原始 944 个动作日中，199 日满足固定恢复必要条件且来源已知；191 日形成成熟局部标签，7 日因次日公司行动、1 日因无次日保留未知。冻结的 1,275 笔 A_0 订单 Tiered 各费用字段复算误差为 0；797 个执行日的原合法投影完全一致，147 个 HOLD 日遵守保存持仓语义。局部 HOLD 分支到次日 09:47 对原账户的持仓、现金、净值误差均为 0。

带方向成交量对局部净优势的 Spearman：全段 0.0989，前后两半 0.2426/0.0272；10,000 次、平均块长 63 日、单侧 0.025/31 的全日历区块下界 -0.0516，不为正。正号及两半方向门虽通过，统计门失败。同期已冻结的 2018–2022 OOF 回归检验在 173/265 条成熟标签上候选 MSE 更差、系数为负、四门全败；本卡不把复用历史的正点估计当作翻案。

区间归因：2024Q3 10 个局部标签的平均恢复价值 -28.09bp，成交量相关 -0.127；2026Q3 49 个标签平均 7.44bp，相关仅 0.019。原 Stage1 在前者的现金保护有效、后者存在现金机会损失；此坐标在关键区间没有可靠区分能力。季度是事后归因，不设交易阈值。

边界：局部共同复位使用次日原账户目标，只为衡量当前动作的条件价值，不是未来可知输入，也不是完整自有状态策略反事实。首小时 bar 的历史首次接收/最终版本未认证；复用历史不构成独立前向。没有新模型拟合或新原生账户。

Qlib Recorder `7abbbcf0fcf145b69df17ee77624aa80` 保存全 944 日日历标签及区块复制值；NAS 归档限定回读 SHA-256 `4b94fb82b73046c56eb477296dd9fd15e8c34ee49187ccef103a6ccd47919013`。下一步保持冻结 Stage1 和 0bp SOXX 硬门，寻找能在 10:32 判断广泛修复与持续下跌的**不同**决策前机制；先查旧恢复 21 项和已关闭盘中/广度路线，再冻结来源与信息门。